# MARS-RL — Mega dataset downloader (Colab / local)
Ek hi notebook run karo → **bada dataset** download + clean + features + zip.

| Block | Data | Source (free, no API key) |
|---|---|---|
| 1 | S&P 500 + DJIA stocks, 2000→ (daily OHLCV, adjusted) | yfinance |
| 2 | Cross-asset ETFs (equity, sector, bonds, gold, oil, dollar, REIT, intl) + crypto | yfinance |
| 3 | Macro: T-bill (risk-free), yields, curve, credit spreads, VIX, oil, dollar, CPI, unemployment, ... | FRED public CSV |
| 4 | Fama-French 5 factors + Momentum (daily) | Kenneth French library |
| 5 | CBOE: VIX, VIX9D, VIX3M, VIX6M, VVIX, SKEW | CBOE CDN |
| 6 | (optional) Earnings dates + quarterly fundamentals | yfinance |
| 7 | (optional, BIG) FNSPID financial news | HuggingFace |
| 8 | Features: 30+ per stock/day, market-state, macro (release-lagged, **no look-ahead**) | computed |

**Runtime:** Colab free ≈ 25–50 min (block 1 sabse zyada). Last cell zip banata hai aur download karta hai.
Pehle `QUICK_TEST=True` se 5 min ka chhota run karke check kar lo, phir `False` karke full run.

In [ ]:
# [CFG] ---------------- CONFIG (yahin badlo) ----------------
QUICK_TEST      = False          # True = sirf ~30 stocks, chhota test
START           = "2000-01-01"
END             = "2024-12-31"   # paper test period 2023-05 tak; extra data optional
MIN_HISTORY_DAYS = 750           # isse kam history wale tickers drop
INCLUDE_ETFS_CRYPTO = True
DOWNLOAD_FUNDAMENTALS = False    # True = earnings dates + quarterly financials (slow, ~1 hr, rate-limited)
DOWNLOAD_NEWS_FNSPID  = False    # True = HuggingFace FNSPID news (multi-GB)
OUT = "mars_data"                # output folder
BATCH = 40                       # tickers per yfinance request

In [ ]:
%pip install -q yfinance stockstats lxml html5lib huggingface_hub pyarrow tqdm requests

In [ ]:
# [DEF] imports + helpers
import os, io, re, time, json, zipfile, warnings, datetime as dt
import numpy as np, pandas as pd, requests
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
for d in ("raw", "processed", "reports"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (research; MARS-RL dataset builder)"}

def get(url, tries=3, **kw):
    for i in range(tries):
        try:
            r = requests.get(url, headers=UA, timeout=60, **kw)
            if r.status_code == 200: return r
        except Exception as e: err = e
        time.sleep(2 * (i + 1))
    raise RuntimeError(f"failed: {url}")

def save_df(df, path_no_ext):
    # parquet agar pyarrow/fastparquet ho, warna csv.gz
    try:
        df.to_parquet(path_no_ext + ".parquet", index=False); return path_no_ext + ".parquet"
    except Exception as e:
        print("parquet unavailable (", type(e).__name__, ") -> saving csv.gz")
        df.to_csv(path_no_ext + ".csv.gz", index=False, compression="gzip"); return path_no_ext + ".csv.gz"

def norm_ticker(t): return str(t).strip().upper().replace(".", "-")

## 1–2. Ticker universe

In [ ]:
# [DEF] universe
DJIA30 = ["AAPL","AMGN","AXP","BA","CAT","CRM","CSCO","CVX","DIS","DOW","GS","HD","HON","IBM","INTC","JNJ","JPM","KO","MCD","MMM","MRK","MSFT","NKE","PG","TRV","UNH","V","VZ","WBA","WMT"]
ETFS = {"equity_broad":["SPY","QQQ","IWM","DIA","MDY","VTI"], "intl":["EFA","EEM","EWJ","FXI","VGK","EWZ"],
        "sectors":["XLB","XLE","XLF","XLI","XLK","XLP","XLU","XLV","XLY","XLC","XLRE"],
        "bonds":["TLT","IEF","SHY","LQD","HYG","TIP","AGG","BND"], "commod":["GLD","SLV","USO","DBC","UNG"],
        "fx_real":["UUP","VNQ"], "vol":["VIXY"]}
CRYPTO = ["BTC-USD","ETH-USD","LTC-USD","XRP-USD","BCH-USD","DOGE-USD","ADA-USD"]

def sp500_tickers():
    try:
        t = pd.read_html(io.StringIO(get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies").text))[0]
        meta = t[["Symbol","Security","GICS Sector","GICS Sub-Industry","Date added"]].copy(); meta.columns = ["ticker","name","sector","industry","date_added"]
    except Exception:
        t = pd.read_csv(io.StringIO(get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv").text))
        meta = t.rename(columns={"Symbol":"ticker","Security":"name","GICS Sector":"sector","GICS Sub-Industry":"industry","Date added":"date_added"})[["ticker","name","sector","industry","date_added"]]
    meta["ticker"] = meta["ticker"].map(norm_ticker); return meta

In [ ]:
meta = sp500_tickers()
stocks = sorted(set(meta.ticker) | set(map(norm_ticker, DJIA30)))
etfs = sorted({t for v in ETFS.values() for t in v}) if INCLUDE_ETFS_CRYPTO else []
crypto = CRYPTO if INCLUDE_ETFS_CRYPTO else []
if QUICK_TEST: stocks, etfs, crypto = stocks[:30], etfs[:5], crypto[:2]
meta.to_csv(f"{OUT}/raw/sp500_constituents_TODAY_survivorship_biased.csv", index=False)
print(len(stocks), "stocks |", len(etfs), "etfs |", len(crypto), "crypto")

## 1–2. Prices (yfinance, split+dividend adjusted)

In [ ]:
# [DEF] price download
import yfinance as yf
def dl_prices(tickers, start, end, batch=BATCH):
    out, failed = [], []
    for i in tqdm(range(0, len(tickers), batch), desc="prices"):
        chunk = tickers[i:i+batch]
        for attempt in range(3):
            try:
                df = yf.download(chunk, start=start, end=end, auto_adjust=True, group_by="ticker", threads=True, progress=False)
                break
            except Exception as e:
                df = None; time.sleep(5 * (attempt + 1))
        if df is None or df.empty: failed += chunk; continue
        for t in chunk:
            try:
                d = df[t] if isinstance(df.columns, pd.MultiIndex) else df
                d = d[["Open","High","Low","Close","Volume"]].dropna(how="all")
                if len(d) == 0: failed.append(t); continue
                d = d.reset_index(); d.columns = ["date","open","high","low","close","volume"]; d["tic"] = t; out.append(d)
            except Exception: failed.append(t)
    return (pd.concat(out, ignore_index=True) if out else pd.DataFrame()), sorted(set(failed))

In [ ]:
px_stocks, f1 = dl_prices(stocks, START, END); px_stocks["asset_class"] = "stock"
parts = [px_stocks]; failed = list(f1)
if etfs:
    p, f2 = dl_prices(etfs, START, END); p["asset_class"] = "etf"; parts.append(p); failed += f2
if crypto:
    p, f3 = dl_prices(crypto, START, END); p["asset_class"] = "crypto"; parts.append(p); failed += f3
raw = pd.concat(parts, ignore_index=True); raw["date"] = pd.to_datetime(raw["date"]).dt.tz_localize(None)
cnt = raw.groupby("tic").size(); keep = cnt[cnt >= MIN_HISTORY_DAYS].index
dropped_short = sorted(set(cnt.index) - set(keep)); raw = raw[raw.tic.isin(keep)].sort_values(["tic","date"]).reset_index(drop=True)
raw = raw[(raw[["open","high","low","close"]] > 0).all(axis=1)]
save_df(raw, f"{OUT}/raw/prices_ohlcv_adjusted_long")
print("rows", len(raw), "| tickers", raw.tic.nunique(), "| failed:", len(failed), "| dropped (short history):", len(dropped_short))
json.dump({"failed": failed, "dropped_short_history": dropped_short}, open(f"{OUT}/reports/price_download_issues.json","w"))

## 3. Macro (FRED public CSV, release-lagged)

In [ ]:
# [DEF] macro
# (series, description, release lag in calendar days applied before use -> avoids look-ahead)
FRED = {"DGS3MO":("3m T-bill yield (risk-free)",1), "DGS2":("2y Treasury",1), "DGS10":("10y Treasury",1), "T10Y2Y":("10y-2y slope",1),
        "T10Y3M":("10y-3m slope",1), "BAA10Y":("Baa-10y credit spread",1), "AAA10Y":("Aaa-10y spread",1), "T10YIE":("10y breakeven inflation",1),
        "VIXCLS":("VIX close",1), "DCOILWTICO":("WTI oil",1), "DTWEXBGS":("Broad USD index",1), "DEXUSEU":("USD/EUR",1),
        "FEDFUNDS":("Fed funds (monthly)",35), "CPIAUCSL":("CPI (monthly)",45), "UNRATE":("Unemployment (monthly)",40),
        "INDPRO":("Industrial production (monthly)",45), "UMCSENT":("Consumer sentiment (monthly)",35),
        "NFCI":("Chicago Fed financial conditions (weekly)",7), "STLFSI4":("St. Louis financial stress (weekly)",7)}
def fred_series(sid):
    r = get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}")
    d = pd.read_csv(io.StringIO(r.text)); d.columns = ["date","value"]
    d["date"] = pd.to_datetime(d["date"]); d["value"] = pd.to_numeric(d["value"], errors="coerce"); return d.dropna()
def build_macro(cal_index, fred):
    cols = {}
    for sid, (desc, lag) in fred.items():
        try:
            d = fred_series(sid); s = d.set_index("date")["value"]; s.index = s.index + pd.Timedelta(days=lag)  # available only after release lag
            cols[sid] = s.reindex(s.index.union(cal_index)).ffill().reindex(cal_index)
        except Exception as e: print("skip", sid, e)
    return pd.DataFrame(cols, index=cal_index)

In [ ]:
cal = pd.DatetimeIndex(sorted(raw.date.unique()))
macro = build_macro(cal, FRED)
macro.to_csv(f"{OUT}/raw/macro_fred_lagged.csv"); print(macro.shape); macro.tail(3)

## 4. Fama-French factors + 5. CBOE indices

In [ ]:
# [DEF] factors + cboe
def french_daily(name, skiprows_marker=None):
    r = get(f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{name}_CSV.zip")
    z = zipfile.ZipFile(io.BytesIO(r.content)); txt = z.read(z.namelist()[0]).decode("latin1").splitlines()
    rows = [l for l in txt if re.match(r"^\s*\d{8}\s*,", l)]
    hdr = next(l for l in txt if "Mkt-RF" in l or "Mom" in l.replace(" ", "") or "WML" in l) if any(("Mkt-RF" in l or "Mom" in l or "WML" in l) for l in txt) else None
    d = pd.read_csv(io.StringIO("\n".join(rows)), header=None); return d, hdr
def build_ff():
    a, _ = french_daily("F-F_Research_Data_5_Factors_2x3_daily"); a.columns = ["date","Mkt_RF","SMB","HML","RMW","CMA","RF"]
    m, _ = french_daily("F-F_Momentum_Factor_daily"); m.columns = ["date","MOM"]
    d = a.merge(m, on="date", how="left"); d["date"] = pd.to_datetime(d["date"].astype(str), format="%Y%m%d")
    d.iloc[:, 1:] = d.iloc[:, 1:] / 100.0; return d   # decimal daily returns; RF = daily risk-free
CBOE = ["VIX","VIX9D","VIX3M","VIX6M","VVIX","SKEW"]
def cboe_idx(n):
    r = get(f"https://cdn.cboe.com/api/global/us_indices/daily_prices/{n}_History.csv")
    d = pd.read_csv(io.StringIO(r.text)); d.columns = [c.strip().upper() for c in d.columns]
    dc = "DATE" if "DATE" in d.columns else d.columns[0]; vc = "CLOSE" if "CLOSE" in d.columns else d.columns[-1]
    d = d[[dc, vc]]; d.columns = ["date", n]; d["date"] = pd.to_datetime(d["date"]); return d.set_index("date")[n]

In [ ]:
try:
    ff = build_ff(); ff.to_csv(f"{OUT}/raw/fama_french_5f_mom_daily.csv", index=False); print("FF", ff.shape, ff.date.min().date(), ff.date.max().date())
except Exception as e: ff = None; print("FF failed:", e)
cb = {}
for n in CBOE:
    try: cb[n] = cboe_idx(n)
    except Exception as e: print("cboe skip", n, e)
cboe = pd.DataFrame(cb).sort_index(); cboe.to_csv(f"{OUT}/raw/cboe_indices_daily.csv"); print("CBOE", cboe.shape)

## 6. (optional) Earnings dates + fundamentals

In [ ]:
if DOWNLOAD_FUNDAMENTALS:
    rows_e, rows_f = [], []
    for t in tqdm(stocks if not QUICK_TEST else stocks[:10], desc="fundamentals"):
        try:
            tk = yf.Ticker(t)
            ed = tk.get_earnings_dates(limit=80)
            if ed is not None and len(ed): ed = ed.reset_index(); ed["tic"] = t; rows_e.append(ed)
            q = tk.quarterly_financials
            if q is not None and q.shape[1]: qq = q.T.reset_index().rename(columns={"index":"period_end"}); qq["tic"] = t; rows_f.append(qq)
        except Exception: pass
        time.sleep(0.3)
    if rows_e: pd.concat(rows_e).to_csv(f"{OUT}/raw/earnings_dates.csv", index=False)
    if rows_f: pd.concat(rows_f).to_csv(f"{OUT}/raw/quarterly_financials_recent.csv", index=False)
    print("fundamentals done (yfinance gives only ~4-5 recent quarters; use SEC XBRL for deep history)")
else: print("skipped (DOWNLOAD_FUNDAMENTALS=False)")

## 7. (optional, BIG) FNSPID financial news

In [ ]:
if DOWNLOAD_NEWS_FNSPID:
    from huggingface_hub import list_repo_files, hf_hub_download
    files = list_repo_files("Zihan1004/FNSPID", repo_type="dataset"); print(files[:30])
    news_files = [f for f in files if "news" in f.lower() and f.lower().endswith((".csv", ".zip", ".parquet"))]
    for f in news_files[:2]:
        p = hf_hub_download("Zihan1004/FNSPID", f, repo_type="dataset", local_dir=f"{OUT}/raw/fnspid"); print("downloaded", p, round(os.path.getsize(p)/1e6), "MB")
else: print("skipped (DOWNLOAD_NEWS_FNSPID=False)")

## 8. Feature engineering (point-in-time: feature(t) uses data ≤ close of day t)

In [ ]:
# [DEF] features
from stockstats import StockDataFrame as Sdf
def stock_features(g):
    g = g.sort_values("date").reset_index(drop=True); c, v = g["close"], g["volume"]
    lr = np.log(c).diff(); f = pd.DataFrame({"date": g["date"], "tic": g["tic"]})
    f["ret1"] = lr
    for k in (5, 10, 20, 60, 120, 252): f[f"ret{k}"] = c / c.shift(k) - 1
    f["mom_12_1"] = c.shift(21) / c.shift(252) - 1                      # 12-1 month momentum
    f["rev5"] = -(c / c.shift(5) - 1)
    for k in (20, 60): f[f"vol{k}"] = lr.rolling(k).std() * np.sqrt(252)
    f["downvol60"] = lr.clip(upper=0).rolling(60).std() * np.sqrt(252)
    f["skew60"] = lr.rolling(60).skew(); f["kurt60"] = lr.rolling(60).kurt()
    for k in (20, 50, 200): f[f"sma{k}_gap"] = c / c.rolling(k).mean() - 1
    f["dist_52w_high"] = c / c.rolling(252).max() - 1; f["dist_52w_low"] = c / c.rolling(252).min() - 1
    f["range20"] = ((g["high"] - g["low"]) / c).rolling(20).mean()
    f["gap_overnight"] = g["open"] / c.shift(1) - 1
    f["dollar_vol20"] = np.log1p((c * v).rolling(20).mean()); f["vol_z60"] = (v - v.rolling(60).mean()) / (v.rolling(60).std() + 1e-9)
    f["amihud20"] = (lr.abs() / (c * v + 1)).rolling(20).mean() * 1e9
    s = Sdf.retype(g[["open","high","low","close","volume"]].copy())
    for ind in ("macd", "rsi_30", "cci_30", "dx_30", "boll_ub", "boll_lb", "atr"):
        try: f[ind] = s[ind].values
        except Exception: f[ind] = np.nan
    f["boll_pos"] = (c - f["boll_lb"]) / (f["boll_ub"] - f["boll_lb"] + 1e-9)
    f["macd_n"] = f["macd"] / c; f["atr_n"] = f["atr"] / c
    return f.drop(columns=["boll_ub", "boll_lb", "macd", "atr"])

def build_features(raw, macro=None, ff=None, cboe=None):
    st = raw[raw.asset_class == "stock"]
    F = pd.concat([stock_features(g) for _, g in tqdm(st.groupby("tic"), desc="features")], ignore_index=True)
    # market-state from equal-weight stock index (past only) + beta to it
    P = st.pivot(index="date", columns="tic", values="close").sort_index(); R = np.log(P).diff()
    ew = R.mean(axis=1); mk = pd.DataFrame(index=P.index)
    mk["mkt_ret1"] = ew; mk["mkt_ret20"] = ew.rolling(20).sum(); mk["mkt_ret60"] = ew.rolling(60).sum()
    mk["mkt_vol20"] = ew.rolling(20).std() * np.sqrt(252); mk["mkt_vol60"] = ew.rolling(60).std() * np.sqrt(252)
    cum = np.exp(ew.fillna(0).cumsum()); mk["mkt_dd252"] = cum / cum.rolling(252, min_periods=1).max() - 1
    mk["breadth50"] = (P > P.rolling(50).mean()).mean(axis=1); mk["disp"] = R.std(axis=1)
    mk["corr_avg60"] = R.rolling(60).corr(ew).mean(axis=1)
    beta = R.rolling(60).cov(ew).div(ew.rolling(60).var(), axis=0)
    F = F.merge(mk.rename_axis("date").reset_index(), on="date", how="left")
    bt = beta.stack().rename("beta60").rename_axis(["date","tic"]).reset_index(); F = F.merge(bt, on=["date","tic"], how="left")
    # cross-sectional ranks (same-day, no look-ahead)
    for c in ("ret20", "ret60", "mom_12_1", "vol60", "rev5", "dollar_vol20"): F[f"{c}_rank"] = F.groupby("date")[c].rank(pct=True)
    if macro is not None:
        m = macro.copy(); m["term_slope_10y3m"] = m.get("DGS10") - m.get("DGS3MO") if "DGS10" in m else np.nan
        F = F.merge(m.add_prefix("fred_").rename_axis("date").reset_index(), on="date", how="left")
    if ff is not None:
        ffl = ff.set_index("date").shift(1).add_prefix("ff_lag1_")      # factor returns known only after the day -> lag 1 (state, not target)
        F = F.merge(ffl.rename_axis("date").reset_index(), on="date", how="left")
    if cboe is not None and len(cboe):
        F = F.merge(cboe.add_prefix("cboe_").rename_axis("date").reset_index(), on="date", how="left")
    F = F.sort_values(["date", "tic"]).reset_index(drop=True)
    num = F.select_dtypes("number").columns; F[num] = F[num].astype("float32")
    return F

In [ ]:
feat = build_features(raw, macro, ff, cboe)
# forward-fill slow/aux series only (never prices), then report NaNs
aux = [c for c in feat.columns if c.startswith(("fred_", "ff_", "cboe_"))]
feat[aux] = feat.groupby("tic")[aux].ffill()
save_df(feat, f"{OUT}/processed/features_stocks_daily")
print(feat.shape); feat.tail(3).T.head(60)

## 9. Data-quality report + zip

In [ ]:
# [DEF] report
def quality(raw, feat):
    q = {"rows_prices": int(len(raw)), "tickers": int(raw.tic.nunique()), "date_min": str(raw.date.min().date()), "date_max": str(raw.date.max().date()),
         "by_asset_class": raw.groupby("asset_class").tic.nunique().to_dict(), "duplicates": int(raw.duplicated(["date","tic"]).sum()),
         "abs_daily_ret_gt_50pct": int((raw.sort_values(["tic","date"]).groupby("tic").close.pct_change().abs() > 0.5).sum()),
         "feature_rows": int(len(feat)), "n_features": int(feat.shape[1] - 2), "feature_nan_share_top10": feat.isna().mean().sort_values(ascending=False).head(10).round(3).to_dict(),
         "survivorship_warning": "S&P 500 list is TODAY's membership -> survivorship bias; disclose in paper",
         "created_utc": dt.datetime.utcnow().isoformat(timespec="seconds")}
    return q

In [ ]:
q = quality(raw, feat); json.dump(q, open(f"{OUT}/reports/data_quality.json", "w"), indent=2); print(json.dumps(q, indent=1))

In [ ]:
zipname = f"{OUT}.zip"
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for f in fs: z.write(os.path.join(root, f))
print(zipname, round(os.path.getsize(zipname) / 1e6), "MB")
try:
    from google.colab import files; files.download(zipname)
except Exception: print("Colab nahi hai -> zip current folder me hai")

## Next step
`mars_data.zip` mujhe (Claude) upload karo. Main naye data par: (1) alpha ka validation IC check, (2) RL ko naye features par chalaunga, (3) wahi locked protocol (validation select → test ek baar).